In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

import config
import pipeline

# ── Display active parameters ─────────────────────────────────────────────────
# Edit config.py to change any of these — no changes to this notebook needed.
print("=" * 55)
print("TARIFF PCE PIPELINE — ACTIVE CONFIGURATION")
print("=" * 55)
rows = [
    ("BEA IO year",             config.IO_YEAR),
    ("Tariff baseline year",    config.TARIFF_BASELINE_YEAR),
    ("Tariff current month",    config.TARIFF_CURRENT_MONTH),
    ("Markup assumption",       config.MARKUP_ASSUMPTION),
    ("Inflation measure",       config.INFLATION_MEASURE),
    ("Counterfactual baseline", config.COUNTERFACTUAL_BASELINE_MONTH),
    ("Excess inflation window", f"{config.EXCESS_BASELINE_START}–{config.EXCESS_BASELINE_END}"),
    ("Core goods categories",   len(config.CORE_GOODS_CATEGORIES)),
]
for label, val in rows:
    print(f"  {label:<30} {val}")
print("=" * 55)


In [ ]:
# ── Step 1: Direct import shares (m_i = imports / total supply) ──────────────
# Methodology §2 / BEA Supply Table 262

import_shares = pipeline.step1_import_shares(config.IO_YEAR, config.BEA_KEY)

print(f"Commodities: {len(import_shares)}")
print(f"Import share range: "
      f"{import_shares['import_share'].min():.4f} – {import_shares['import_share'].max():.4f}")
print(f"\nTop 10 by direct import share:")
print(
    import_shares.nlargest(10, 'import_share')
    [['BEA_code', 'BEA_descr', 'import_share']]
    .to_string(index=False)
)


   Key                                                        Desc
0   59        Total Requirements, Commodity-by-Commodity - Summary
1   58         Total Requirements, Commodity-by-Commodity - Sector
2   57         Total Requirements, Industry-by-Commodity - Summary
3   56          Total Requirements, Industry-by-Commodity - Sector
4   61          Total Requirements, Industry-by-Industry - Summary
5   60           Total Requirements, Industry-by-Industry - Sector
6  262  The Domestic Supply of Commodities by Industries - Summary
7  261   The Domestic Supply of Commodities by Industries - Sector
8  259              The Use of Commodities by Industries - Summary
9  258               The Use of Commodities by Industries - Sector


In [ ]:
# ── Steps 2–3: Technical coefficients (A) and Leontief inverse (L) ───────────
# Methodology §3–4 / BEA Use Table 259

industries, A, L = pipeline.step2_3_leontief(config.IO_YEAR, config.BEA_KEY)

I_n = np.eye(len(industries))
print(f"Industries: {len(industries)}")
print(f"A matrix shape:                  {A.shape}")
print(f"Max A column sum (must be < 1):  {A.sum(axis=0).max():.4f}")
print(f"Condition number of (I - A):     {np.linalg.cond(I_n - A):.1f}")
print(f"Leontief diagonal mean (> 1):    {np.diag(L).mean():.4f}")
print(f"L min value (should be ≥ 0):     {L.min():.4f}")


(54, 3)
   BEA_code                                  BEA_descr  imports
0     111CA                                      Farms    58047
1     113FF  Forestry, fishing, and related activities    24155
2       211                     Oil and gas extraction   218966
3       212                 Mining, except oil and gas     7202
4       213              Support activities for mining     1354
5        22                                  Utilities     4700
6     311FT     Food and beverage and tobacco products   147959
7     313TT    Textile mills and textile product mills    40020
8     315AL    Apparel and leather and allied products   160497
9       321                              Wood products    36508
10      322                             Paper products    32328
11      323    Printing and related support activities     3277
12      324                Petroleum and coal products    94183
13      325                          Chemical products   365975
14      326               Plasti

In [ ]:
# ── Step 4: Total import content (m_total = m' L) ────────────────────────────
# Methodology §5
# Supply-chain amplification: how much indirect imports add on top of direct shares.

m_vec, m_total = pipeline.step4_total_import_content(import_shares, industries, L)

print(f"Direct import shares mean:   {m_vec.mean():.4f}")
print(f"Total import content mean:   {m_total.mean():.4f}")
print(f"Supply-chain amplification:  {m_total.mean() / m_vec.mean():.2f}×")
print(f"\nSample — direct vs. total import content (first 15 industries):")
print(f"  {'Industry':<14}  {'Direct':>8}  {'Total':>8}  {'Ratio':>6}")
for i, ind in enumerate(industries[:15]):
    ratio = m_total[i] / m_vec[i] if m_vec[i] > 0 else float('nan')
    print(f"  {ind:<14}  {m_vec[i]:8.4f}  {m_total[i]:8.4f}  {ratio:6.2f}×")


(54, 5)
  BEA_code                                  BEA_descr  imports  total_supply  \
0    111CA                                      Farms    58047        640105   
1    113FF  Forestry, fishing, and related activities    24155         99263   
2      211                     Oil and gas extraction   218966        754690   
3      212                 Mining, except oil and gas     7202        114751   
4      213              Support activities for mining     1354        126089   
5       22                                  Utilities     4700        856286   
6    311FT     Food and beverage and tobacco products   147959       1351897   
7    313TT    Textile mills and textile product mills    40020         87431   
8    315AL    Apparel and leather and allied products   160497        179069   
9      321                              Wood products    36508        212101   

   import_share  
0      0.090684  
1      0.243343  
2      0.290140  
3      0.062762  
4      0.010738  
5  

In [ ]:
# ── Step 5: Δτ = τ_current − τ_baseline per BEA industry ─────────────────────
# Methodology §6
# Baseline: annual average over all 12 months of config.TARIFF_BASELINE_YEAR
# Current:  single month config.TARIFF_CURRENT_MONTH

delta_tariff_df = pipeline.step5_delta_tariff(
    config.IMPORTS_FILE,
    config.TARIFF_BASELINE_YEAR,
    config.TARIFF_CURRENT_MONTH,
)

print(f"BEA industries: {len(delta_tariff_df)}")
print(f"\nTop 10 by tariff increase (Δτ):")
print(
    delta_tariff_df.nlargest(10, 'delta_tariff')
    [['bea_io', 'bea_desc', 'tau_base', 'tau', 'delta_tariff']]
    .to_string(index=False)
)


(4642, 10)
TableID      object
Year         object
RowCode      object
RowDescr     object
RowType      object
ColCode      object
ColDescr     object
ColType      object
DataValue    object
NoteRef      object
dtype: object
int64
count    4.642000e+03
mean     1.051351e+05
std      1.241760e+06
min     -1.520810e+05
25%      1.542500e+02
50%      1.362500e+03
75%      9.602500e+03
max      5.018778e+07
Name: DataValue, dtype: float64


In [ ]:
# ── PCE Bridge + Step 6: Predicted tariff effect per PCE category ─────────────
# Methodology §7
# Markup assumption (config.MARKUP_ASSUMPTION):
#   "constant_dollar"  → numerator weighted by producers' value (Fed conservative baseline)
#   "constant_percent" → numerator weighted by purchasers' value (larger consumer effect)

pce_bridge    = pipeline.load_pce_bridge(config.IO_YEAR, config.BEA_KEY)
pce_effect_df = pipeline.step6_pce_effect(
    industries      = industries,
    m_vec           = m_vec,
    L               = L,
    delta_tariff_df = delta_tariff_df,
    pce_bridge      = pce_bridge,
    markup          = config.MARKUP_ASSUMPTION,
)

print(f"Markup assumption: {config.MARKUP_ASSUMPTION}")
print(f"\nTop 15 PCE categories by predicted tariff effect:")
print(pce_effect_df.head(15)[['PCE_category', 'predicted_effect']].to_string(index=False))

# Weighted-average effect on core goods — carried forward to counterfactual cells
core = pce_effect_df.query('PCE_category in @config.CORE_GOODS_CATEGORIES')
core_effect = (
    (core['predicted_effect'] * core['purchasers_value_total']).sum()
    / core['purchasers_value_total'].sum()
)
print(f"\nWeighted avg predicted effect — core goods: {core_effect:.2%}")


Industries for A matrix: 68
A matrix shape: (68, 68)
Max column sum (should be < 1): 0.9079


In [ ]:
# ── Step 7a: Monthly counterfactual (NIPA T20804) ────────────────────────────
# Methodology §8
# config.INFLATION_MEASURE: "core_pce" (PCE ex food & energy) or "headline_pce"
# For the core-goods quarterly index, see Cell 8.

if config.INFLATION_MEASURE == "core_goods_pce":
    print("INFLATION_MEASURE = 'core_goods_pce' — skipping monthly counterfactual.")
    print("The quarterly core-goods counterfactual is computed in Cell 8.")
else:
    baseline_yr = int(config.COUNTERFACTUAL_BASELINE_MONTH[:4])
    current_yr  = int(config.TARIFF_CURRENT_MONTH[:4])
    years       = sorted({baseline_yr, current_yr})

    inflation_series = pipeline.step7_load_inflation(
        config.INFLATION_MEASURE, config.BEA_KEY, years
    )
    result = pipeline.step7_counterfactual(
        inflation_series      = inflation_series,
        pce_effect_df         = pce_effect_df,
        baseline_month        = config.COUNTERFACTUAL_BASELINE_MONTH,
        current_month         = config.TARIFF_CURRENT_MONTH,
        core_goods_categories = config.CORE_GOODS_CATEGORIES,
    )

    label = "Core PCE" if config.INFLATION_MEASURE == "core_pce" else "Headline PCE"
    print(f"{label}  ({config.COUNTERFACTUAL_BASELINE_MONTH} → {config.TARIFF_CURRENT_MONTH})")
    print(f"  Actual inflation:            {result['actual_inflation']:.2%}")
    print(f"  Core goods tariff effect:    {result['core_goods_effect']:.2%}")
    print(f"  Core goods share of PCE:     {result['core_goods_share']:.2%}")
    print(f"  Tariff contribution to PCE:  {result['tariff_contribution']:.2%}")
    print(f"  Counterfactual (no tariffs): {result['counterfactual_inflation']:.2%}")


L shape: (68, 68)
Min value (should be >= 0): 0.0000
Diagonal mean (should be > 1): 1.1124


In [ ]:
# ── Step 7b: Quarterly core-goods price index + counterfactual (NIPA T20404) ──
# Methodology §8
# Constructs a PCE-weighted price index for the 27 core goods categories and
# computes the counterfactual 2024Q4 → 2025Q4 inflation (no tariff scenario).

baseline_yr = int(config.COUNTERFACTUAL_BASELINE_MONTH[:4])
current_yr  = int(config.TARIFF_CURRENT_MONTH[:4])
years       = sorted({baseline_yr, current_yr})

core_goods_index = pipeline.step7_core_goods_index(
    api_key               = config.BEA_KEY,
    years                 = years,
    pce_effect_df         = pce_effect_df,
    core_goods_categories = config.CORE_GOODS_CATEGORIES,
    nipa_crosswalk        = config.NIPA_CROSSWALK,
)

baseline_q = f"{baseline_yr}Q4"
current_q  = f"{current_yr}Q4"
idx_base   = core_goods_index[baseline_q]
idx_latest = core_goods_index[current_q]

actual_cg_inflation = (idx_latest - idx_base) / idx_base
counterfactual_cg   = actual_cg_inflation - core_effect

print("Core goods price index (quarterly):")
print(core_goods_index.to_string())
print(f"\nActual core goods inflation ({baseline_q} → {current_q}):  {actual_cg_inflation:.2%}")
print(f"Predicted tariff contribution:                          {core_effect:.2%}")
print(f"Counterfactual (no tariffs):                            {counterfactual_cg:.2%}")


m_total shape: (68,)
Direct import shares mean:  0.1026
Total import content mean:  0.2232
Sample - direct vs total:
  111CA: direct=0.0907, total=0.2643
  113FF: direct=0.2433, total=0.3368
  211: direct=0.2901, total=0.4381
  212: direct=0.0628, total=0.1808
  213: direct=0.0107, total=0.0951
  22: direct=0.0055, total=0.0902
  23: direct=0.0000, total=0.1956
  311FT: direct=0.1094, total=0.3332
  313TT: direct=0.4577, total=0.7946
  315AL: direct=0.8963, total=1.0369
  321: direct=0.1721, total=0.4112
  322: direct=0.1307, total=0.3563
  323: direct=0.0423, total=0.2223
  324: direct=0.0957, total=0.4031
  325: direct=0.2899, total=0.4692
  326: direct=0.2179, total=0.4945
  327: direct=0.1833, total=0.3340
  331: direct=0.2856, total=0.5233
  332: direct=0.1802, total=0.4342
  333: direct=0.3496, total=0.6458


In [ ]:
# ── Step 7c: Predicted tariff effect vs. excess inflation — scatter plot ──────
# Methodology §9
# Excess inflation = current-year YoY growth minus mean of baseline-window YoY growth.
# Baseline window: config.EXCESS_BASELINE_START – config.EXCESS_BASELINE_END (inclusive).
# To adjust, change those two values in config.py.

current_year = int(config.TARIFF_CURRENT_MONTH[:4])

summary_df = pipeline.step7_excess_inflation(
    api_key               = config.BEA_KEY,
    pce_effect_df         = pce_effect_df,
    core_goods_categories = config.CORE_GOODS_CATEGORIES,
    nipa_crosswalk        = config.NIPA_CROSSWALK,
    current_year          = current_year,
    baseline_start        = config.EXCESS_BASELINE_START,
    baseline_end          = config.EXCESS_BASELINE_END,
)

print(
    summary_df[['PCE_category', 'predicted_effect', 'excess_inflation', 'pce_share']]
    .to_string(index=False)
)

# ── Scatter plot ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))
plot_df = summary_df.dropna(subset=['predicted_effect', 'excess_inflation', 'pce_share'])

ax.scatter(
    plot_df['predicted_effect'] * 100,
    plot_df['excess_inflation'] * 100,
    s=plot_df['pce_share'] * 50000,
    alpha=0.5, color='darkblue', edgecolors='black', linewidths=0.5,
)

x = plot_df['predicted_effect'].values * 100
y = plot_df['excess_inflation'].values * 100
w = plot_df['pce_share'].values

wls_model  = sm.WLS(y, sm.add_constant(x), weights=w).fit()
intercept, slope = wls_model.params
print(wls_model.summary())

x_fit = np.linspace(x.min() - 1.5, x.max() + 1.5, 100)
ax.plot(x_fit, slope * x_fit + intercept,
        color='firebrick', linewidth=2, linestyle='--',
        label=f'WLS fit (slope={slope:.2f})')

ax.axhline(0, color='black', linewidth=1.5, linestyle='--', alpha=0.7)
ax.axvline(0, color='black', linewidth=1.5, linestyle='--', alpha=0.7)
ax.set_xlim(-0.5, 7.0)

baseline_range = f"{config.EXCESS_BASELINE_START}–{config.EXCESS_BASELINE_END}"
ax.set_title(
    f"Tariff-Predicted vs. Excess Inflation by Core Goods Category\n"
    f"(bubble size = PCE share, baseline {baseline_range})",
    fontsize=20,
)
ax.set_xlabel("Predicted tariff effect (pp)", fontsize=16)
ax.set_ylabel(f"Excess inflation vs. {baseline_range} trend (pp)", fontsize=16)
ax.legend(fontsize=12, frameon=False, loc='upper right')
ax.tick_params(axis='x', labelsize=20)
ax.tick_params(axis='y', labelsize=20)
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

plt.tight_layout()
plt.savefig("tariff_vs_excess_inflation.png", bbox_inches="tight", dpi=150)
plt.show()


(297, 10)
   NIPA_line                           PCE_category commodity_code  \
0          5                    New motor vehicles          3361MV   
1          6  Net purchases of used motor vehicles            Used   
2          7   Motor vehicle parts and accessories             325   

                                  commodity_descr  producers_value  \
0  Motor vehicles, bodies and trailers, and parts           201291   
1                Scrap, used and secondhand goods           102734   
2                               Chemical products              223   

   transport_costs  wholesale  retail  purchasers_value  year  
0             3864       9638  160288            375081  2022  
1             2624       3282  128873            237514  2022  
2                5         29     141               398  2022  
['NIPA_line', 'PCE_category', 'commodity_code', 'commodity_descr', 'producers_value', 'transport_costs', 'wholesale', 'retail', 'purchasers_value', 'year']
PCE_category
Men